# Segmentação de clientes com aprendizado não supervisionado

**Mini Projeto — IA / Machine Learning: Clustering**

**Autores:** Nicolas Mariano da Silva (RA 00333111) e Pedro Henrique Isamu Fagundes de Souza Tsukahara Yoshissaro (RA 00330820; @knutzin)

Este notebook explora perfis de clientes de shopping usando idade, renda anual e pontuação de gastos. Comparamos K-Means, clustering aglomerativo e DBSCAN.

> O conjunto é didático e não representa clientes reais. Os resultados demonstram técnicas de clustering e não devem orientar decisões comerciais reais.


## 1. Pergunta e objetivo

**Pergunta:** é possível identificar perfis distintos de clientes usando características demográficas e uma pontuação de gastos?

Vamos inspecionar e preparar os dados, visualizar variáveis, comparar algoritmos e descrever os grupos. Como não há rótulos verdadeiros, a avaliação será interna: silhouette, inspeção visual e coerência dos perfis. Isso não comprova que os segmentos sejam “verdadeiros”.


## 2. Dados e origem

Fonte: [Mall Customer Segmentation Data — Kaggle](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python). O CSV em data acompanha o projeto para execução reproduzível. O dataset tem identificador, gênero, idade, renda anual (milhares de dólares) e pontuação de gastos (1–100).

O identificador será removido da modelagem. Usaremos idade, renda e pontuação de gastos. Gênero aparece na EDA, mas fica fora do clustering para evitar impor uma distância arbitrária a uma variável categórica binária. As três variáveis numéricas serão padronizadas.


In [ ]:
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', palette='Set2')
RANDOM_STATE = 42
df = pd.read_csv(Path('Mall_Customers.csv'))
df.head()


## 3. Análise exploratória

Verificamos dimensões, tipos, valores ausentes, duplicatas e estatísticas descritivas. Depois visualizamos distribuições, relações e correlações.


In [ ]:
numeric_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
print(f'Linhas: {df.shape[0]} | Colunas: {df.shape[1]}')
display(df.dtypes.to_frame('tipo'))
display(df.isna().sum().to_frame('valores ausentes'))
print(f'Linhas duplicadas: {df.duplicated().sum()}')
display(df.describe(include='all').T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax, color='#4C78A8')
    ax.set_title(f'Distribuição: {col}')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Genre', ax=axes[0])
sns.scatterplot(data=df, x='Age', y='Spending Score (1-100)', hue='Genre', ax=axes[1])
axes[0].set_title('Renda × gastos')
axes[1].set_title('Idade × gastos')
plt.tight_layout()
plt.show()
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='vlag', center=0, fmt='.2f')
plt.title('Correlação entre variáveis numéricas')
plt.show()


## 4. Preparação das variáveis

Removemos CustomerID e padronizamos idade, renda anual e pontuação de gastos. Gênero foi analisado visualmente, mas não usado no modelo porque a codificação binária impõe uma distância arbitrária no espaço euclidiano.


In [ ]:
features = df[numeric_cols].copy()
if features.isna().any().any():
    raise ValueError('Há valores ausentes nas variáveis selecionadas')
X = StandardScaler().fit_transform(features)
print('Formato da matriz modelada:', X.shape)
display(features.head())


## 5. K-Means: escolha de k

Calculamos inércia (método do cotovelo) e silhouette para k de 2 a 10. O cotovelo é uma heurística; o maior silhouette não garante a melhor segmentação para todo objetivo.


In [ ]:
k_values = range(2, 11)
inertias, silhouettes = [], []
for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = model.fit_predict(X)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X, labels))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_values), inertias, marker='o')
axes[0].set(title='Método do cotovelo', xlabel='k', ylabel='Inércia')
axes[1].plot(list(k_values), silhouettes, marker='o', color='#F58518')
axes[1].set(title='Silhouette médio', xlabel='k', ylabel='Silhouette')
plt.tight_layout()
plt.show()
scores_k = pd.DataFrame({'k': list(k_values), 'inercia': inertias, 'silhouette': silhouettes})
display(scores_k.round(3))
best_k = int(scores_k.loc[scores_k['silhouette'].idxmax(), 'k'])
print(f'k escolhido pelo maior silhouette: {best_k}')


## 6. Ajuste e comparação dos métodos

Comparamos K-Means e clustering aglomerativo com o mesmo k e testamos DBSCAN. Para DBSCAN, calculamos silhouette somente quando existem pelo menos dois clusters sem ruído; pontos -1 são ruído e ficam fora da métrica.


In [ ]:
labels_kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20).fit_predict(X)
labels_agg = AgglomerativeClustering(n_clusters=best_k, linkage='ward').fit_predict(X)
labels_dbscan = DBSCAN(eps=0.65, min_samples=5).fit_predict(X)

def cluster_summary(name, labels):
    labels = np.asarray(labels)
    keep = labels != -1
    n_clusters = len(set(labels[keep]))
    score = silhouette_score(X[keep], labels[keep]) if n_clusters >= 2 else np.nan
    return {'método': name, 'clusters sem ruído': n_clusters,
            'ruído (%)': 100*np.mean(labels == -1), 'silhouette': score}
display(pd.DataFrame([
    cluster_summary('K-Means', labels_kmeans),
    cluster_summary('Aglomerativo (Ward)', labels_agg),
    cluster_summary('DBSCAN (eps=0.65, min_samples=5)', labels_dbscan)
]).round(3))


## 7. PCA e interpretação dos segmentos

PCA reduz as quatro variáveis preparadas a duas componentes para facilitar a visualização. A projeção pode ocultar separações em outras dimensões e não substitui as métricas do espaço completo.


In [ ]:
X_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, title, labels in zip(axes, ['K-Means', 'Aglomerativo', 'DBSCAN'],
                              [labels_kmeans, labels_agg, labels_dbscan]):
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab10', s=35, alpha=.8)
    ax.set_title(title)
    ax.set_xlabel('Componente principal 1')
    ax.set_ylabel('Componente principal 2')
plt.tight_layout()
plt.show()

result = df[numeric_cols].copy()
result['cluster'] = labels_kmeans
profile = result.groupby('cluster').agg(
    clientes=('cluster', 'size'),
    idade_media=('Age', 'mean'),
    renda_media_k=('Annual Income (k$)', 'mean'),
    gastos_medios=('Spending Score (1-100)', 'mean')
).round(1)
display(profile.sort_values('gastos_medios', ascending=False))
sns.scatterplot(data=result, x='Annual Income (k$)', y='Spending Score (1-100)',
                hue='cluster', palette='tab10', s=55)
plt.title('Perfis K-Means: renda e pontuação de gastos')
plt.show()


## 8. Conclusões e limitações

Na execução com `random_state=42`, o maior silhouette entre k=2 e k=10 foi **0,427 em k=6**. A inércia cai com k, com uma desaceleração visível por volta de k=5–6; o silhouette e o cotovelo apontam para uma solução compacta nessa faixa. Esse resultado depende das variáveis, da padronização e do intervalo pesquisado.

Com k=6, K-Means teve silhouette **0,427** e o método aglomerativo Ward, **0,420**. DBSCAN com `eps=0.65` e `min_samples=5` encontrou **2 clusters**, classificou **10,5%** dos registros como ruído e obteve silhouette **0,265** (ruído excluído do cálculo). Nesta configuração, DBSCAN separou menos claramente os grupos; seus resultados podem mudar bastante com os parâmetros.

Os perfis K-Means incluem, por exemplo, um grupo de renda alta e gastos baixos (média de renda 88,9 mil e gastos 17,0) e outro de renda alta e gastos altos (86,5 mil e 82,1). Há também um grupo jovem de renda mais baixa e gastos altos (idade média 25,2; renda 25,8 mil; gastos 76,9). Esses nomes são descrições aproximadas dos centróides, não categorias naturais nem recomendações de marketing.

**Limitações:** base pequena, didática e sem rótulos externos; pontuação de gastos simplificada; padronização, algoritmo e parâmetros influenciam os resultados; silhouette avalia separação geométrica, não utilidade de negócio. Não generalize os perfis para clientes reais.

**Próximos passos:** testar sensibilidade a parâmetros, outras variáveis e validar segmentos com dados representativos e contexto de negócio, respeitando privacidade e vieses.

## Referências

- Choudhary, V. *Mall Customer Segmentation Data*. Kaggle. https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
- Scikit-learn. *Clustering*. https://scikit-learn.org/stable/modules/clustering.html
- Scikit-learn. *Silhouette Coefficient*. https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html